# Soil Moisture Classification Notebook

This notebook shows the full step-by-step workflow for your project:

1. Load the CSV data
2. Inspect the columns
3. Decide the feature and target
4. Split the data into training and testing sets
5. Train a model
6. Test the model
7. Save the trained model


## Why `raw` is the feature and `status` is the target

- `raw` is the actual soil moisture sensor reading.
- `status` is the output label with three classes: `GREEN`, `RED`, and `YELLOW`.
- `pump_on` is not a good feature because it is already derived from the status and would leak the answer.
- `time_valid` is always `True` in this dataset, so it does not help the model learn.
- `reading_id`, `ts_epoch`, `ts_iso`, and `ts_ms` are record IDs or time fields, not real moisture features.


In [1]:
from pathlib import Path
import pickle

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text


In [2]:
# Find the CSV whether the notebook is run from the ML_model folder or the project root.
CSV_PATH = None
for candidate in (Path("moisture_readings.csv"), Path("ML_model/moisture_readings.csv")):
    if candidate.exists():
        CSV_PATH = candidate
        break

if CSV_PATH is None:
    raise FileNotFoundError("Could not find moisture_readings.csv")

SAVE_DIR = CSV_PATH.parent
MODEL_PATH = SAVE_DIR / "moisture_status_model.pkl"
PREDICTIONS_PATH = SAVE_DIR / "model_predictions.csv"

print("CSV file:", CSV_PATH.resolve())
print("Model save path:", MODEL_PATH.resolve())
print("Predictions save path:", PREDICTIONS_PATH.resolve())


CSV file: C:\Users\pinki\OneDrive\Desktop\8thcollegproject\ML_model\moisture_readings.csv
Model save path: C:\Users\pinki\OneDrive\Desktop\8thcollegproject\ML_model\moisture_status_model.pkl
Predictions save path: C:\Users\pinki\OneDrive\Desktop\8thcollegproject\ML_model\model_predictions.csv


## Step 1: Load the dataset


In [3]:
df = pd.read_csv(CSV_PATH)
df.head()


,reading_id,pump_on,raw,status,time_valid,ts_epoch,ts_iso,ts_ms
0,1700001001,True,2870,RED,True,1700001001,2026-02-05 14:10:01,12345678
1,1770287965,False,4095,YELLOW,True,1770287965,2026-02-05 10:39:25,260115
2,1770287968,False,4095,YELLOW,True,1770287968,2026-02-05 10:39:28,263494
3,1770287970,False,4095,YELLOW,True,1770287970,2026-02-05 10:39:30,264723
4,1770287971,False,4095,YELLOW,True,1770287971,2026-02-05 10:39:31,266167


## Step 2: Inspect the data


In [4]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)


Dataset shape: (38794, 8)

Columns:
['reading_id', 'pump_on', 'raw', 'status', 'time_valid', 'ts_epoch', 'ts_iso', 'ts_ms']

Data types:
reading_id     int64
pump_on         bool
raw            int64
status        object
time_valid      bool
ts_epoch       int64
ts_iso        object
ts_ms          int64
dtype: object


In [5]:
print("Missing values in each column:")
print(df.isnull().sum())

print("\nUnique status values:")
print(df["status"].unique())

print("\nUnique values in time_valid:")
print(df["time_valid"].unique())


Missing values in each column:
reading_id    0
pump_on       0
raw           0
status        0
time_valid    0
ts_epoch      0
ts_iso        0
ts_ms         0
dtype: int64

Unique status values:
['RED' 'YELLOW' 'GREEN']

Unique values in time_valid:
[ True]


In [6]:
print("Status distribution:")
print(df["status"].value_counts())

print("\nPump state by status:")
print(pd.crosstab(df["status"], df["pump_on"]))

print("\nRaw reading summary by status:")
print(df.groupby("status")["raw"].agg(["min", "max", "mean", "count"]))


Status distribution:
status
GREEN     25882
YELLOW    11527
RED        1385
Name: count, dtype: int64

Pump state by status:
pump_on  False  True 
status               
GREEN    25882      0
RED          0   1385
YELLOW   11527      0

Raw reading summary by status:
         min   max         mean  count
status                                
GREEN   1039  2498  1915.983232  25882
RED     2502  3583  3006.167509   1385
YELLOW  3600  4095  4071.554611  11527


## Step 3: Select feature and target

We use only `raw` as the input feature and `status` as the output target.


In [7]:
X = df[["raw"]]
y = df["status"]

print("Feature columns:", X.columns.tolist())
print("Target column:", y.name)
print("\nFeature sample:")
X.head()


Feature columns: ['raw']
Target column: status

Feature sample:


,raw
0,2870
1,4095
2,4095
3,4095
4,4095


## Step 4: Split into training and testing data


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 31035
Testing samples: 7759


## Step 5: Train the model

A decision tree works well here because the moisture classes are separated by threshold ranges.


In [9]:
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

print("Model training completed.")


Model training completed.


## Step 6: Test the model


In [10]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred))


Accuracy: 0.9999

Classification report:
              precision    recall  f1-score   support

       GREEN       1.00      1.00      1.00      5177
         RED       1.00      1.00      1.00       277
      YELLOW       1.00      1.00      1.00      2305

    accuracy                           1.00      7759
   macro avg       1.00      1.00      1.00      7759
weighted avg       1.00      1.00      1.00      7759



In [11]:
cm_df = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=model.classes_),
    index=model.classes_,
    columns=model.classes_
)

print("Confusion matrix:")
cm_df


Confusion matrix:


,GREEN,RED,YELLOW
GREEN,5177,0,0
RED,1,276,0
YELLOW,0,0,2305


In [12]:
results = X_test.copy()
results["actual_status"] = y_test.values
results["predicted_status"] = y_pred
results["correct"] = results["actual_status"] == results["predicted_status"]

results.head(20)


,raw,actual_status,predicted_status,correct
12877,4095,YELLOW,YELLOW,True
9752,4095,YELLOW,YELLOW,True
25098,1661,GREEN,GREEN,True
26237,2034,GREEN,GREEN,True
10872,4095,YELLOW,YELLOW,True
5824,2032,GREEN,GREEN,True
23840,1685,GREEN,GREEN,True
27120,1863,GREEN,GREEN,True
34469,1667,GREEN,GREEN,True
8365,2318,GREEN,GREEN,True


## Step 7: Read the learned rules


In [13]:
print(export_text(model, feature_names=["raw"]))


|--- raw <= 2504.50
|   |--- class: GREEN
|--- raw >  2504.50
|   |--- raw <= 3591.50
|   |   |--- class: RED
|   |--- raw >  3591.50
|   |   |--- class: YELLOW



## Step 8: Test the model on new sample readings


In [14]:
sample_readings = pd.DataFrame({"raw": [1500, 2600, 3200, 3700, 4095]})
sample_readings["predicted_status"] = model.predict(sample_readings)
sample_readings["predicted_pump_on"] = sample_readings["predicted_status"].eq("RED")

sample_readings


,raw,predicted_status,predicted_pump_on
0,1500,GREEN,False
1,2600,RED,True
2,3200,RED,True
3,3700,YELLOW,False
4,4095,YELLOW,False


## Step 9: Save the trained model and prediction file


In [15]:
results_to_save = X_test.copy()
results_to_save["actual_status"] = y_test.values
results_to_save["predicted_status"] = y_pred
results_to_save["predicted_pump_on"] = results_to_save["predicted_status"].eq("RED")

results_to_save.to_csv(PREDICTIONS_PATH, index=False)

with open(MODEL_PATH, "wb") as model_file:
    pickle.dump(model, model_file)

print("Saved model to:", MODEL_PATH.resolve())
print("Saved predictions to:", PREDICTIONS_PATH.resolve())


Saved model to: C:\Users\pinki\OneDrive\Desktop\8thcollegproject\ML_model\moisture_status_model.pkl
Saved predictions to: C:\Users\pinki\OneDrive\Desktop\8thcollegproject\ML_model\model_predictions.csv
